# AI-13 · L15 · Notebook D: memory keeps growing (GPU out of memory after many steps)

This notebook contains a broken training setup from Lesson 15 (Common Training Problems and Fixes). It runs on a CPU runtime; no GPU is needed.

**Symptom you should see:** The training loop runs and the loss falls, but the process uses more and more memory at every step. On a GPU runtime this ends in `CUDA out of memory` after enough steps; on CPU the same problem shows as RAM use that keeps rising (the loop prints it every 5 steps, and you can also watch the Colab resource panel). A correct loop should level off after the first few steps; here it climbs by several MB per step.

**Your task:** run every cell unchanged, write the symptom in one line, then use the L15 debugging checklist (shapes, dtypes and devices; overfit one small batch; loss and label format; learning rate; gradient norms; memory) to find the cause. Change one thing at a time and record the evidence that your fix worked.

## Setup
Imports, a fixed random seed and the data.

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
print("PyTorch", torch.__version__)
from torch.utils.data import TensorDataset, DataLoader
import os, resource, sys

def ram_mb():
    """Current resident memory (RAM) of this process in MB."""
    try:  # Linux, including Colab
        return int(open("/proc/self/statm").read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
    except OSError:  # macOS / other: fall back to peak memory
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak / 2**20 if sys.platform == "darwin" else peak / 1024

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Synthetic Fashion-MNIST-shaped data: 12,800 flattened 28x28 "images", 10 classes
X = torch.randn(12800, 784)
y = torch.randint(0, 10, (12800,))
train_dl = DataLoader(TensorDataset(X, y), batch_size=256, shuffle=True)

model = nn.Sequential(nn.Linear(784, 2048), nn.ReLU(),
                      nn.Linear(2048, 2048), nn.ReLU(),
                      nn.Linear(2048, 10)).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

## Training code (broken)
Run this cell unchanged first and note the symptom.

In [ ]:
# --- Code as given in the lesson; the "..." is filled in with a training step and a log line ---
losses = []
for step, (xb, yb) in enumerate(train_dl):
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad()
    loss_fn(model(xb), yb).backward()
    opt.step()
    loss = loss_fn(model(xb), yb)                # batch loss after the update, for the log
    losses.append(loss)                          # keeps every graph alive
    if step % 5 == 0:
        mem = f"GPU {torch.cuda.memory_allocated() / 2**20:.0f} MB" if device == "cuda" else f"RAM {ram_mb():.0f} MB"
        print(f"step {step:2d}  loss {loss.item():.3f}  {mem}")

## Your fix
Copy the code above into a new cell, change one thing at a time, and show the evidence that it now trains correctly.

In [ ]:
# Your fixed version here
